In [30]:
print("Data Validation Notebook")

Data Validation Notebook


# Import Libraries

In [31]:
import pandas as pd
import numpy as np

# Load Data

In [32]:
DATA_PATH = "../data/raw/HHS_Unaccompanied_Alien_Children_Program.csv"
df = pd.read_csv(DATA_PATH)

In [33]:
df

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,"December 21, 2025",6.0,18.0,11.0,"2,484",14.0
1,"December 18, 2025",11.0,50.0,6.0,"2,472",16.0
2,"December 17, 2025",7.0,31.0,11.0,"2,481",10.0
3,"December 16, 2025",8.0,54.0,15.0,"2,468",9.0
4,"December 15, 2025",11.0,42.0,9.0,"2,470",7.0
...,...,...,...,...,...,...
1165,NaN,NaN,NaN,NaN,NaN,NaN
1166,NaN,NaN,NaN,NaN,NaN,NaN
1167,NaN,NaN,NaN,NaN,NaN,NaN
1168,NaN,NaN,NaN,NaN,NaN,NaN


# Rename Columns for easy readability

In [34]:
df.columns = [
    "date",
    "cbp_daily_intake",
    "cbp_custody",
    "cbp_transfers_to_hhs",
    "hhs_care",
    "hhs_discharges",
]

In [35]:
df

,date,cbp_daily_intake,cbp_custody,cbp_transfers_to_hhs,hhs_care,hhs_discharges
0,"December 21, 2025",6.0,18.0,11.0,"2,484",14.0
1,"December 18, 2025",11.0,50.0,6.0,"2,472",16.0
2,"December 17, 2025",7.0,31.0,11.0,"2,481",10.0
3,"December 16, 2025",8.0,54.0,15.0,"2,468",9.0
4,"December 15, 2025",11.0,42.0,9.0,"2,470",7.0
...,...,...,...,...,...,...
1165,NaN,NaN,NaN,NaN,NaN,NaN
1166,NaN,NaN,NaN,NaN,NaN,NaN
1167,NaN,NaN,NaN,NaN,NaN,NaN
1168,NaN,NaN,NaN,NaN,NaN,NaN


# Convert Date + Sort

In [36]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Rule : Time-series data = always sorted

In [37]:
df

,date,cbp_daily_intake,cbp_custody,cbp_transfers_to_hhs,hhs_care,hhs_discharges
0,2023-01-12,33.0,53.0,34.0,"6,566",436.0
1,2023-01-22,32.0,49.0,39.0,"7,122",227.0
2,2023-01-23,32.0,50.0,39.0,"7,280",181.0
3,2023-01-24,47.0,42.0,47.0,"7,433",175.0
4,2023-01-25,20.0,22.0,41.0,"7,538",180.0
...,...,...,...,...,...,...
1165,NaT,NaN,NaN,NaN,NaN,NaN
1166,NaT,NaN,NaN,NaN,NaN,NaN
1167,NaT,NaN,NaN,NaN,NaN,NaN
1168,NaT,NaN,NaN,NaN,NaN,NaN


# changing data types

In [38]:
numeric_cols = [
    "cbp_daily_intake",
    "cbp_custody",
    "cbp_transfers_to_hhs",
    "hhs_care",
    "hhs_discharges",
]

for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("—", "", regex=False)
        .str.replace("NA", "", regex=False)
        .str.replace("N/A", "", regex=False)
        .str.strip()
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [39]:
df.dtypes

date                    datetime64[ns]
cbp_daily_intake               float64
cbp_custody                    float64
cbp_transfers_to_hhs           float64
hhs_care                       float64
hhs_discharges                 float64
dtype: object

# Check Duplicate Dates

In [40]:
duplicate_dates = df[df.duplicated(subset=["date"], keep=False)]
print("Duplicate date records:", duplicate_dates.shape[0])
duplicate_dates.head()

Duplicate date records: 450


,date,cbp_daily_intake,cbp_custody,cbp_transfers_to_hhs,hhs_care,hhs_discharges
720,NaT,NaN,NaN,NaN,NaN,NaN
721,NaT,NaN,NaN,NaN,NaN,NaN
722,NaT,NaN,NaN,NaN,NaN,NaN
723,NaT,NaN,NaN,NaN,NaN,NaN
724,NaT,NaN,NaN,NaN,NaN,NaN


# Check Missing Dates

# small intution of this works

- Tumne pd.date_range se ek poora calendar banaya hai — start se end tak har ek din.
- Phir us calendar ko compare kiya df["date"] ke saath.
- Jo dates tumhari DataFrame mein nahi mili, woh missing_dates mein aa gayi.
- Output mein jo DatetimeIndex([...]) dikh raha hai, woh tumhe bata raha hai: “In dates pe tumhare data mein entry hi nahi hai.”


In [41]:
full_date_range = pd.date_range(start=df["date"].min(), end=df["date"].max(), freq="D")
full_date_range

DatetimeIndex(['2023-01-12', '2023-01-13', '2023-01-14', '2023-01-15',
               '2023-01-16', '2023-01-17', '2023-01-18', '2023-01-19',
               '2023-01-20', '2023-01-21',
               ...
               '2025-12-12', '2025-12-13', '2025-12-14', '2025-12-15',
               '2025-12-16', '2025-12-17', '2025-12-18', '2025-12-19',
               '2025-12-20', '2025-12-21'],
              dtype='datetime64[ns]', length=1075, freq='D')

In [42]:
missing_dates = full_date_range.difference(df["date"])
print("Missing dates count:", len(missing_dates))
missing_dates[:10]

Missing dates count: 355


DatetimeIndex(['2023-01-13', '2023-01-14', '2023-01-15', '2023-01-16',
               '2023-01-17', '2023-01-18', '2023-01-19', '2023-01-20',
               '2023-01-21', '2023-01-26'],
              dtype='datetime64[ns]', freq=None)

# Logical Constraint Validation

* Rule-1 : Transfers ≤ CBP Custody
* Rule-2 : Discharges ≤ HHS Care

In [43]:
invalid_cbp_transfer = df[df["cbp_transfers_to_hhs"] > df["cbp_custody"]]

print("Invalid CBP transfer records:", invalid_cbp_transfer.shape[0])
invalid_cbp_transfer.head()

Invalid CBP transfer records: 86


,date,cbp_daily_intake,cbp_custody,cbp_transfers_to_hhs,hhs_care,hhs_discharges
3,2023-01-24,47.0,42.0,47.0,7433.0,175.0
4,2023-01-25,20.0,22.0,41.0,7538.0,180.0
9,2023-02-02,15.0,13.0,23.0,7879.0,298.0
22,2023-02-22,107.0,215.0,230.0,7978.0,232.0
23,2023-02-23,101.0,162.0,178.0,7914.0,386.0


In [44]:
df['hhs_care']

0       6566.0
1       7122.0
2       7280.0
3       7433.0
4       7538.0
         ...  
1165       NaN
1166       NaN
1167       NaN
1168       NaN
1169       NaN
Name: hhs_care, Length: 1170, dtype: float64

In [45]:
invalid_hhs_discharge = df[df["hhs_discharges"] > df["hhs_care"]]
print("Invalid HHS discharge records:", invalid_hhs_discharge.shape[0])
invalid_hhs_discharge.head()

Invalid HHS discharge records: 0


,date,cbp_daily_intake,cbp_custody,cbp_transfers_to_hhs,hhs_care,hhs_discharges


# Create Anomaly Flags (Best Practice)

In [46]:
df["flag_invalid_cbp_transfer"] = df["cbp_transfers_to_hhs"] > df["cbp_custody"]
df["flag_invalid_hhs_discharge"] = df["hhs_discharges"] > df["hhs_care"]

In [47]:
df

,date,cbp_daily_intake,cbp_custody,cbp_transfers_to_hhs,hhs_care,hhs_discharges,flag_invalid_cbp_transfer,flag_invalid_hhs_discharge
0,2023-01-12,33.0,53.0,34.0,6566.0,436.0,False,False
1,2023-01-22,32.0,49.0,39.0,7122.0,227.0,False,False
2,2023-01-23,32.0,50.0,39.0,7280.0,181.0,False,False
3,2023-01-24,47.0,42.0,47.0,7433.0,175.0,True,False
4,2023-01-25,20.0,22.0,41.0,7538.0,180.0,True,False
...,...,...,...,...,...,...,...,...
1165,NaT,NaN,NaN,NaN,NaN,NaN,False,False
1166,NaT,NaN,NaN,NaN,NaN,NaN,False,False
1167,NaT,NaN,NaN,NaN,NaN,NaN,False,False
1168,NaT,NaN,NaN,NaN,NaN,NaN,False,False


# Negative Value Check

In [ ]:
negative_values = (
    df[
        [
            "cbp_daily_intake",
            "cbp_custody",
            "cbp_transfers_to_hhs",
            "hhs_care",
            "hhs_discharges",
        ]
    ]
    < 0
).sum()

negative_values

cbp_daily_intake        0
cbp_custody             0
cbp_transfers_to_hhs    0
hhs_care                0
hhs_discharges          0
dtype: int64

# Validation Summary Table

In [49]:
validation_summary = {
    "total_records": len(df),
    "duplicate_dates": duplicate_dates.shape[0],
    "missing_dates": len(missing_dates),
    "invalid_cbp_transfers": invalid_cbp_transfer.shape[0],
    "invalid_hhs_discharges": invalid_hhs_discharge.shape[0],
}

pd.DataFrame.from_dict(validation_summary, orient="index", columns=["count"])

,count
total_records,1170
duplicate_dates,450
missing_dates,355
invalid_cbp_transfers,86
invalid_hhs_discharges,0


### Data Validation Findings

- Date continuity was evaluated across the full reporting window.
- Logical constraints were enforced through anomaly flags rather than data removal.
- Identified anomalies were preserved to maintain reporting transparency.
- Dataset is suitable for feature engineering and trend analysis.
